In [1]:
import numpy as np
import networkx as nx
from collections import defaultdict

In [2]:
def A_test(A):
    boo = True
    n = 2*(A.shape[0]-2)
    if np.sum(A)!=3*n:
        boo = False
        #print('Test 1 failed')
    if np.sum(np.sum(A,axis=0)==5)!=12:
        boo = False
        #print('Test 2 failed')
    if np.sum(np.sum(A,axis=0)==6)!=int(n/2-10):
        boo = False
        #print('Test 3 failed')
    return boo

def spiral_to_A(n,penta_indexes,to_print=False):
    m = int(n/2+2)
    A = np.zeros((m,m))

    # Create a counter for each vertex how many connenctions are left
    free_edge_counter = np.zeros(m)
    for i in range(m):
        if np.sum(i==penta_indexes)==1:
            free_edge_counter[i] = 5
        else:
            free_edge_counter[i] = 6
    ####print(Free_edge_counter)
    #Spiral conjecture statement (1):
    # Each new face in the spiral after the second shares an edge with both its immediate predecessor in the spiral and ....
    for i in range(m-1):
        A[i,i+1] = 1
        A[i+1,i] = 1
    free_edge_counter[0] = free_edge_counter[0]-1
    free_edge_counter[-1] = free_edge_counter[-1]-1
    free_edge_counter[1:-1] = free_edge_counter[1:-1]-2
    ####print(Free_edge_counter)


    #Spiral conjecture statement (2):
    #the first face first_open_face in the preceding spiral that still has an open edge
    new_face = 0
    open_face = 2
    left_overs_start = False
    cuts = np.ones((m,2))*m
    cuts[:,1] = np.arange(0,m)
    cuts_counter = 0
    while np.sum(free_edge_counter)>2 and not left_overs_start:

        # Basic step
        if free_edge_counter[new_face]>0 and free_edge_counter[open_face]>0 and A[new_face,open_face]==0:
            #if to_print:
                #print(new_face,open_face)
            A[new_face,open_face] = 1
            A[open_face,new_face] = 1
            free_edge_counter[new_face] -= 1
            free_edge_counter[open_face] -= 1
            open_face += 1
        #Case: 'Cut corner' 
        #new_face is complete => Check all later faces whether they are complete, and if so, 
        #connected their precursor and successor
        if free_edge_counter[new_face] == 0:
            if new_face < m-1:
                for i in range(new_face+1,m-2):
                    if free_edge_counter[i]==0 and len(np.where(free_edge_counter[0:i]>0)[0])>0 and len(np.where(free_edge_counter[i:m]>0)[0])>0:
                        # Find largest precursor and smallest successor who are both not complete:
                        #print(i)
                        max_pre = np.max(np.where(free_edge_counter[0:i]>0)[0])
                        min_suc = i + np.min(np.where(free_edge_counter[i:m]>0)[0])
                        if A[max_pre,min_suc]==0:
                            #if to_print:
                            #    print('Case: Cut corner')
                            #    print(max_pre,min_suc)
                            cuts[min_suc,:] = int(np.min([max_pre,cuts[min_suc,0]])),int(min_suc)
                            cuts_counter += 1
                            #if to_print:
                            #    print('----------------')
                            A[max_pre,min_suc] = 1
                            A[min_suc,max_pre] = 1
                            free_edge_counter[max_pre] -= 1
                            free_edge_counter[min_suc] -= 1
                            if np.sum(free_edge_counter)<3:
                                left_overs_start = True
            if not left_overs_start:
                new_face = np.min(np.where(free_edge_counter>0)[0])
                if cuts_counter > 0 and cuts[new_face,0] < m:
                    open_face = np.max(np.where(A[int(cuts[new_face,0]),:]==1)[0])
                else:
                    open_face = np.max(np.where(A[new_face-1,:]==1)[0])
    
    left_overs = np.where(free_edge_counter)[0]
    if len(left_overs)==2:
    #    if to_print:
    #        print(left_overs[0],left_overs[1])
        A[left_overs[0],left_overs[1]] = 1
        A[left_overs[1],left_overs[0]] = 1
        free_edge_counter[left_overs[0]] -= 1
        free_edge_counter[left_overs[1]] -= 1
    #if to_print:
    #    print(A_test(A))
    return A  

def A_split_A5_A6(A):
    # Input: A is the adjacency matrix of a dual fullerene graph T_n
    penta_indices = np.where(np.sum(A,axis=1)==5)[0]
    A5 = A[penta_indices,:]
    A5 = A5[:,penta_indices]
    A6 = np.delete(A, penta_indices, axis=0)
    A6 = np.delete(A6, penta_indices, axis=1)
    return A5,A6

def count_non_unique_rows(matrix):
    """
    Computes the number of non-unique rows in a two-dimensional NumPy array.

    Parameters:
        matrix (numpy.ndarray): A 2D array.

    Returns:
        int: The number of non-unique rows.
    """
    # Convert rows to tuples for easy comparison
    row_tuples = [tuple(row) for row in matrix]

    # Count occurrences of each row using numpy's unique
    unique_rows, counts = np.unique(row_tuples, axis=0, return_counts=True)

    # Count rows that occur more than once
    non_unique_count = np.sum(counts > 1)

    for j in range(np.shape(matrix)[0]-1):
        temp = matrix[j,:]
        for jj in range(j+1,np.shape(matrix)[0]):
            if tuple(temp) == tuple(matrix[jj,:]):
                print(j,jj)
            

    return non_unique_count

def all_NP(A):
    k_max = np.shape(A)[0]
    result = np.zeros(k_max+1)
    for k in range(k_max+1):
        result[k] = np.trace(np.linalg.matrix_power(A,k))
    return result

